# Ground-Truth Review & Export

Upload a **slim Excel** (`query`, `actual_output`/`expected_output`, `conversation_history`,
`extracted_metadata`), review each row as a **dark card** — query, answer (Markdown), the
**circular images** right beneath for comparison, then metadata — tick the good ones, and
**Generate CSV**.

**Output CSV columns:** `query` (`{"query": ...}`), `expected_output` (`{"answer": ...}`),
`conversation_history` (JSON array or `[]`), `metadata` (circulars JSON array).
UTF-8 with BOM so Arabic opens correctly in Excel.

## 1. Setup, credentials & options

In [ ]:
# %pip install ipywidgets pandas openpyxl httpx markdown --quiet
import pandas as pd, csv, json, base64, mimetypes, datetime as dt, html as _html
from pathlib import Path
import httpx
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- credentials for authenticated image fetch (Approach B) ---
LANGFUSE_PUBLIC_KEY = "pk-lf-xxxxxxxx"
LANGFUSE_SECRET_KEY = "sk-lf-xxxxxxxx"

# --- options ---
IMG_WIDTH = 900               # rendered image width (px)
DROP_IMG_URL_IN_CSV = True    # omit img_url from the exported metadata column (cards still show images)

# --- dark theme palette ---
C_CARD   = "#1e2330"   # card background
C_PANEL  = "#262c3b"   # inner panels
C_BORDER = "#3a4256"   # borders
C_TEXT   = "#e8ecf4"   # primary text (soft off-white)
C_MUTED  = "#9aa4ba"   # labels / secondary
C_ACCENT = "#7aa2ff"   # headings / accents
C_BTN    = "#5b8def"   # primary button
C_OK     = "#3fbf8f"   # positive

OUT_DIR = Path("ground_truth_exports"); OUT_DIR.mkdir(exist_ok=True)

_img_client = httpx.Client(auth=(LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY),
                           verify=False, trust_env=False,
                           timeout=httpx.Timeout(30.0, read=60.0), follow_redirects=True)
_img_cache = {}

def fetch_image_datauri(url):
    if not url: return None
    if url in _img_cache: return _img_cache[url]
    try:
        r = _img_client.get(url); r.raise_for_status()
        ctype = (r.headers.get("content-type", "").split(";")[0]
                 or mimetypes.guess_type(url)[0] or "image/png")
        uri = f"data:{ctype};base64,{base64.b64encode(r.content).decode()}"
        _img_cache[url] = uri
        return uri
    except Exception as e:
        print(f"  ! image fetch failed ({type(e).__name__}) for {url[:80]}")
        return None

def _detect_columns(df):
    cols = {c.strip().lower(): c for c in df.columns}
    qcol = cols.get("query") or cols.get("input")
    acol = cols.get("actual_output") or cols.get("expected_output") or cols.get("answer")
    hcol = cols.get("conversation_history") or cols.get("history")
    mcol = cols.get("extracted_metadata") or cols.get("metadata")
    if qcol is None: qcol = list(df.columns)[0]
    return qcol, acol, hcol, mcol

def _wrap_query(text):
    if text is None: text = ""
    s = str(text).strip()
    if s.startswith("{"):
        try:
            o = json.loads(s)
            while isinstance(o, dict) and "query" in o:
                o = o["query"]
                if isinstance(o, str) and o.strip().startswith("{") and '"query"' in o:
                    try: o = json.loads(o.strip())
                    except Exception: break
            return json.dumps({"query": o}, ensure_ascii=False)
        except Exception: pass
    return json.dumps({"query": text}, ensure_ascii=False)

def _wrap_answer(text):
    if text is None: text = ""
    s = str(text).strip()
    if s.startswith("{"):
        try:
            o = json.loads(s)
            while isinstance(o, dict) and "answer" in o:
                o = o["answer"]
                if isinstance(o, str) and o.strip().startswith("{") and '"answer"' in o:
                    try: o = json.loads(o.strip())
                    except Exception: break
            return json.dumps({"answer": o}, ensure_ascii=False)
        except Exception: pass
    return json.dumps({"answer": text}, ensure_ascii=False)

def _norm_history(val):
    if val is None: return "[]"
    s = str(val).strip()
    if s in ("", "[]"): return "[]"
    try:
        o = json.loads(s)
        if isinstance(o, list): return json.dumps(o, ensure_ascii=False) if o else "[]"
        return json.dumps(o, ensure_ascii=False)
    except Exception: return s

def _parse_circulars(val):
    if val is None: return []
    s = str(val).strip()
    if s in ("", "[]"): return []
    try:
        o = json.loads(s)
        if isinstance(o, list): return [c for c in o if isinstance(c, dict)]
    except Exception: pass
    out = []
    for line in s.split("\n"):
        parts = [p.strip() for p in line.split("|")]
        if len(parts) >= 1 and parts[0]:
            keys = ["timestamp", "filename", "attachment_id", "summary", "img_url"]
            out.append({keys[i]: parts[i] for i in range(min(len(parts), len(keys)))})
    return out

def _md_render(text):
    raw = "" if text is None else str(text)
    try:
        import markdown as _mdlib
        return _mdlib.markdown(raw, extensions=["nl2br", "tables", "sane_lists"])
    except Exception:
        return _html.escape(raw).replace("\n", "<br>")

print("Setup ready. Exports ->", OUT_DIR.resolve())

## 2. Choose the Excel file
Use the uploader, **or** set `EXCEL_PATH` in cell 3.

In [ ]:
uploader = widgets.FileUpload(accept=".xlsx", multiple=False, description="Upload slim .xlsx")
display(uploader)
print("After picking a file, run cell 3. (Or set EXCEL_PATH in cell 3.)")

## 3. Load the rows

In [ ]:
EXCEL_PATH = ""   # optional: e.g. "langfuse_exports/xxx_slim.xlsx"

df = None
if EXCEL_PATH:
    df = pd.read_excel(EXCEL_PATH); print("Loaded:", EXCEL_PATH)
elif uploader.value:
    up = uploader.value[0] if isinstance(uploader.value, (list, tuple)) else list(uploader.value.values())[0]
    content = up["content"] if isinstance(up, dict) else up.content
    import io
    df = pd.read_excel(io.BytesIO(content))
    print("Loaded upload:", up.get("name") if isinstance(up, dict) else getattr(up, "name", "uploaded.xlsx"))
else:
    raise ValueError("No file. Use the uploader in cell 2, or set EXCEL_PATH here.")

df = df.fillna("")
QCOL, ACOL, HCOL, MCOL = _detect_columns(df)
print(f"Rows: {len(df)}")
print(f"Detected -> query: {QCOL!r} | answer: {ACOL!r} | history: {HCOL!r} | metadata: {MCOL!r}")
df.head(3)

## 4. Review cards (dark)

Per card: **query + answer** (Markdown) → **circular images** (full-width toggle, lazy + cached,
fold/unfold) → **metadata** → **history** (collapsed). Tick the keepers.

Most of each card is rendered as one styled HTML block so the dark theme is consistent
regardless of your Jupyter theme; the checkbox, image toggle, and accordion are real widgets.

In [ ]:
row_checks = []

def _esc(t):
    return (str(t) if t is not None else "").replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")

def _panel(text, size="14px", mono=False):
    fam = "ui-monospace,Menlo,Consolas,monospace" if mono else "Segoe UI,Tahoma,Arial,sans-serif"
    return (f"<div dir='rtl' style='text-align:right;white-space:pre-wrap;font-size:{size};"
            f"font-family:{fam};line-height:1.8;color:{C_TEXT};background:{C_PANEL};"
            f"border:1px solid {C_BORDER};border-radius:8px;padding:10px 12px;margin:6px 0'>{_esc(text)}</div>")

def _md_panel(text, size="14.5px"):
    return (f"<div dir='rtl' style='text-align:right;font-size:{size};color:{C_TEXT};"
            f"font-family:Segoe UI,Tahoma,Arial,sans-serif;line-height:1.85;background:{C_PANEL};"
            f"border:1px solid {C_BORDER};border-right:3px solid {C_ACCENT};border-radius:8px;"
            f"padding:10px 12px;margin:6px 0'>{_md_render(text)}</div>")

def _label(txt):
    return (f"<div style='color:{C_MUTED};font-size:11px;font-weight:700;letter-spacing:.06em;"
            f"text-transform:uppercase;margin:4px 0 0'>{txt}</div>")

def _circular_table(circs):
    if not circs:
        return f"<i style='color:{C_MUTED}'>No circulars</i>"
    out = []
    for idx, c in enumerate(circs):
        rows = []
        for label, key in [("التاريخ / timestamp","timestamp"),("الملف / filename","filename"),
                           ("attachment_id","attachment_id"),("الملخص / summary","summary")]:
            v = c.get(key)
            if v: rows.append(f"<tr><td style='color:{C_MUTED};padding:3px 10px;white-space:nowrap;"
                              f"vertical-align:top;font-size:12px'>{label}</td>"
                              f"<td dir='rtl' style='color:{C_TEXT};padding:3px 10px;font-size:13px'>{_esc(v)}</td></tr>")
        out.append(f"<div style='color:{C_ACCENT};font-weight:700;margin:8px 0 4px;font-size:13px'>● Circular {idx+1}</div>"
                   f"<table style='border-collapse:collapse;width:100%;background:{C_PANEL};"
                   f"border:1px solid {C_BORDER};border-radius:8px'>{''.join(rows)}</table>")
    return "".join(out)

def _query_text(v):
    s = str(v).strip()
    if s.startswith("{"):
        try: return json.loads(s).get("query", s)
        except Exception: return s
    return s
def _answer_text(v):
    s = str(v).strip()
    if s.startswith("{"):
        try: return json.loads(s).get("answer", s)
        except Exception: return s
    return s

def build_review():
    global row_checks
    row_checks = []
    blocks = []
    for i in range(len(df)):
        q = df.iloc[i][QCOL]
        a = df.iloc[i][ACOL] if ACOL else ""
        hist = _norm_history(df.iloc[i][HCOL]) if HCOL else "[]"
        circs = _parse_circulars(df.iloc[i][MCOL]) if MCOL else []
        try: n_turns = len(json.loads(hist))
        except Exception: n_turns = 0

        chip = (f"<span style='background:{C_BTN};color:#fff;border-radius:20px;padding:2px 10px;"
                f"font-size:11px;font-weight:700;margin-right:8px'>{len(circs)} circular(s)</span>") if circs else ""
        # header + query + answer as ONE styled block (top of card)
        top_html = widgets.HTML(
            f"<div style='display:flex;align-items:center;gap:8px;margin-bottom:8px'>"
            f"<span style='background:{C_ACCENT};color:#0f131c;border-radius:6px;padding:3px 11px;"
            f"font-weight:800;font-size:13px'>Row {i}</span>{chip}</div>"
            f"<div style='display:flex;gap:14px;flex-wrap:wrap'>"
            f"<div style='flex:1;min-width:300px'>{_label('Query')}{_panel(_query_text(q))}</div>"
            f"<div style='flex:1;min-width:300px'>{_label('Answer → expected_output')}{_md_panel(_answer_text(a))}</div>"
            f"</div>")

        chk = widgets.Checkbox(value=False, description="Keep this row", indent=False)
        row_checks.append(chk)

        img_out = widgets.Output()
        toggle = widgets.Button(
            description=(f"Show {len(circs)} circular image(s)" if circs else "No circular images"),
            disabled=(len(circs) == 0), icon="image",
            layout=widgets.Layout(width="100%", height="44px"))
        toggle.style.button_color = C_BTN
        try:
            toggle.style.text_color = "#ffffff"; toggle.style.font_weight = "700"
        except Exception: pass
        toggle._loaded = False; toggle._visible = False

        def _make_toggle(circs_local, out_local, btn_local):
            def _click(_):
                if not btn_local._loaded:
                    btn_local.disabled = True; btn_local.description = "Loading images…"
                    with out_local:
                        clear_output()
                        for j, c in enumerate(circs_local):
                            url = c.get("img_url")
                            if not url:
                                display(HTML(f"<i style='color:{C_MUTED}'>Circular {j+1}: no img_url</i>")); continue
                            uri = fetch_image_datauri(url)
                            if uri:
                                display(HTML(
                                    f"<div style='margin:8px 0'>"
                                    f"<div style='color:{C_ACCENT};font-weight:700;font-size:12px;margin-bottom:4px'>Circular {j+1}</div>"
                                    f"<img src='{uri}' style='max-width:{IMG_WIDTH}px;width:100%;height:auto;"
                                    f"border:1px solid {C_BORDER};border-radius:10px'></div>"))
                            else:
                                display(HTML(f"<i style='color:#ff8080'>Circular {j+1}: image failed to load</i>"))
                    btn_local._loaded = True; btn_local._visible = True; btn_local.disabled = False
                    btn_local.description = f"Hide {len(circs_local)} image(s)"
                else:
                    btn_local._visible = not btn_local._visible
                    out_local.layout.display = "" if btn_local._visible else "none"
                    btn_local.description = (f"Hide {len(circs_local)} image(s)" if btn_local._visible
                                             else f"Show {len(circs_local)} image(s)")
            return _click
        toggle.on_click(_make_toggle(circs, img_out, toggle))

        meta_html = widgets.HTML(_label("Circular metadata") + _circular_table(circs))

        hist_acc = widgets.Accordion(children=[widgets.HTML(_panel(hist if n_turns else "[]", size="12.5px", mono=True))])
        hist_acc.set_title(0, f"Conversation history ({n_turns} turn(s))")
        hist_acc.selected_index = None

        inner = widgets.VBox([top_html, chk, toggle, img_out, meta_html, hist_acc])
        card = widgets.VBox([inner], layout=widgets.Layout(
            border=f"1px solid {C_BORDER}", border_radius="14px", padding="16px 18px", margin="14px 0"))
        card.add_class(f"gtcard{i}")
        blocks.append((card, i))

    # one CSS block (plain string, NOT an f-string for the brace-heavy parts)
    rules = []
    for _, idx in blocks:
        rules.append(".gtcard%d { background: %s !important; box-shadow: 0 4px 18px rgba(0,0,0,0.30); }" % (idx, C_CARD))
    extra = (
        "[class*='gtcard'] .widget-checkbox label, [class*='gtcard'] .widget-label { color: %s !important; font-weight:600; }"
        % C_TEXT
        + " .jupyter-widget-Collapse-header, .p-Collapse-header { background: %s !important; color: %s !important; border-radius:8px !important; }"
        % (C_PANEL, C_MUTED)
    )
    css = widgets.HTML("<style>" + " ".join(rules) + " " + extra + "</style>")

    sel_all = widgets.Button(description="Select all", icon="check"); sel_all.style.button_color = C_OK
    clr_all = widgets.Button(description="Clear all", icon="times"); clr_all.style.button_color = "#3a4153"
    try:
        sel_all.style.text_color = "#ffffff"; clr_all.style.text_color = C_TEXT
    except Exception: pass
    counter = widgets.HTML()
    def _refresh(_=None):
        n = sum(c.value for c in row_checks)
        counter.value = (f"<span style='color:{C_TEXT};font-size:15px'>"
                         f"<b style='color:{C_ACCENT};font-size:18px'>{n}</b> / {len(row_checks)} selected</span>")
    sel_all.on_click(lambda _: [setattr(c, "value", True) for c in row_checks])
    clr_all.on_click(lambda _: [setattr(c, "value", False) for c in row_checks])
    for c in row_checks: c.observe(_refresh, names="value")
    _refresh()
    bar = widgets.HBox([sel_all, clr_all, counter],
                       layout=widgets.Layout(padding="10px 14px", margin="0 0 6px 0",
                                            border=f"1px solid {C_BORDER}", border_radius="10px"))
    bar.add_class("gtbar")
    barcss = widgets.HTML("<style>.gtbar { background: %s !important; box-shadow:0 2px 10px rgba(0,0,0,0.25); }</style>" % C_CARD)
    display(css, barcss, bar, widgets.VBox([c for c, _ in blocks]))

build_review()

## 5. Generate the ground-truth CSV

In [ ]:
gen_btn = widgets.Button(description="Generate CSV", icon="download",
                         layout=widgets.Layout(width="220px", height="42px"))
gen_btn.style.button_color = "#5b8def"
try: gen_btn.style.text_color = "#ffffff"; gen_btn.style.font_weight = "700"
except Exception: pass
gen_out = widgets.Output()

def _generate(_):
    with gen_out:
        clear_output()
        idx = [i for i, c in enumerate(row_checks) if c.value]
        if not idx:
            print("No rows selected. Tick at least one row in cell 4."); return
        src = df.iloc[idx].copy()
        records = []
        for _, r in src.iterrows():
            rec = {"query": _wrap_query(r[QCOL] if QCOL else "")}
            if ACOL: rec["expected_output"] = _wrap_answer(r[ACOL])
            rec["conversation_history"] = _norm_history(r[HCOL]) if HCOL else "[]"
            if MCOL:
                _circs = _parse_circulars(r[MCOL])
                if DROP_IMG_URL_IN_CSV:
                    _circs = [{k: v for k, v in c.items() if k != "img_url"} for c in _circs]
                rec["metadata"] = json.dumps(_circs, ensure_ascii=False)
            records.append(rec)
        col_order = [c for c in ["query", "expected_output", "conversation_history", "metadata"]
                     if c in records[0]]
        out = pd.DataFrame(records)[col_order]
        stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
        path = OUT_DIR / f"ground_truth_{stamp}.csv"
        out.to_csv(path, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)
        print(f"Wrote {len(out)} row(s) -> {path}")
        print("Columns:", list(out.columns), "| img_url in metadata:", not DROP_IMG_URL_IN_CSV)
        display(out.head(min(10, len(out))))

gen_btn.on_click(_generate)
display(gen_btn, gen_out)

---
### Notes
- **Dark theme**: each card's text is rendered inside styled HTML so colors hold regardless of
  your Jupyter theme; card backgrounds use injected CSS keyed to per-card classes. Tune the
  `C_*` palette in cell 1.
- **Card order**: query + answer → circular images (toggle, lazy + cached) → metadata →
  history (collapsed).
- **Answer** = Markdown with line breaks (`nl2br`). Needs `markdown`; falls back to escaped
  text with `<br>` otherwise.
- **`DROP_IMG_URL_IN_CSV`**: strips `img_url` from the exported metadata; cards still show images.
- **CSV**: `query` (`{"query": ...}`), `expected_output` (`{"answer": ...}`),
  `conversation_history`, `metadata`. UTF-8 with BOM. Wrapping is idempotent (peels nesting).